# First imports

In [23]:
from __future__ import annotations

import importlib
from pathlib import Path

import numpy as np
import pandas as pd

# Extraction d'exiobase

In [ ]:
exio = importlib.reload(importlib.import_module("exiobase_io"))

## Sans endogénéisation du capital

In [28]:
DATA_FOLDER = "Data/"
YEAR = 2019
SYSTEM = "pxp"
FRANCE = "FR"
OUTPUT_FOLDER = Path("Results")

RE_IMPORT_EXIOBASE = True      # True = rebuild the feathers from the ZIPs
ENDOGENEISER_CAPITAL = False


EXIO_PATH_ZIP = DATA_FOLDER + "Source files/IOT_2019_pxp.zip"


d = exio.generate_exiobase(
    DATA_FOLDER, YEAR, SYSTEM,
    exio_path_zip=EXIO_PATH_ZIP,
    re_import_exiobase=RE_IMPORT_EXIOBASE,
    endogenise_capital=ENDOGENEISER_CAPITAL,
    indicator=None,
    satellite=None
)

Z, A, Y, x, _, _, _ = (d[k] for k in ["Z", "A", "Y", "x", "F", "F_Y", "Kbar"])

Parsing EXIOBASE (newest)...
  ✓ EXIOBASE (newest) parsed


### Calcul de la matrice de Leontief

In [29]:
L = np.linalg.inv(np.eye(A.shape[0]) - A.values)

## Exiobase regions

In [ ]:
regions = Z.index.get_level_values("region").unique()
print(f"{len(regions)} regions:")
print(list(regions))

49 regions:
['AT', 'BE', 'BG', 'CY', 'CZ', 'DE', 'DK', 'EE', 'ES', 'FI', 'FR', 'GR', 'HR', 'HU', 'IE', 'IT', 'LT', 'LU', 'LV', 'MT', 'NL', 'PL', 'PT', 'RO', 'SE', 'SI', 'SK', 'GB', 'US', 'JP', 'CN', 'CA', 'KR', 'BR', 'IN', 'MX', 'RU', 'AU', 'CH', 'TR', 'TW', 'NO', 'ID', 'ZA', 'WA', 'WL', 'WE', 'WF', 'WM']


In [ ]:
exiobase_regions = {
    # --- EU27 ---
    "AT": "Austria",
    "BE": "Belgium",
    "BG": "Bulgaria",
    "CY": "Cyprus",
    "CZ": "Czechia",
    "DE": "Germany",
    "DK": "Denmark",
    "EE": "Estonia",
    "ES": "Spain",
    "FI": "Finland",
    "FR": "France",
    "GR": "Greece",
    "HR": "Croatia",
    "HU": "Hungary",
    "IE": "Ireland",
    "IT": "Italy",
    "LT": "Lithuania",
    "LU": "Luxembourg",
    "LV": "Latvia",
    "MT": "Malta",
    "NL": "Netherlands",
    "PL": "Poland",
    "PT": "Portugal",
    "RO": "Romania",
    "SE": "Sweden",
    "SI": "Slovenia",
    "SK": "Slovakia",
    # --- Other individual countries ---
    "GB": "United Kingdom",
    "US": "United States",
    "JP": "Japan",
    "CN": "China",
    "CA": "Canada",
    "KR": "South Korea",
    "BR": "Brazil",
    "IN": "India",
    "MX": "Mexico",
    "RU": "Russia",
    "AU": "Australia",
    "CH": "Switzerland",
    "TR": "Turkey",
    "TW": "Taiwan",
    "NO": "Norway",
    "ID": "Indonesia",
    "ZA": "South Africa",
    # --- Rest of World aggregates ---
    "WA": "Rest of World - Asia and Pacific",
    "WL": "Rest of World - Latin America",
    "WE": "Rest of World - Europe",
    "WF": "Rest of World - Africa",
    "WM": "Rest of World - Middle East",
}

## Exiobase products

In [23]:
sectors = Z.index.get_level_values("sector").unique()

print(f"\n{len(sectors)} sectors/products:")
for s in sectors:
    print(" -", s)


200 sectors/products:
 - Paddy rice
 - Wheat
 - Cereal grains nec
 - Vegetables, fruit, nuts
 - Oil seeds
 - Sugar cane, sugar beet
 - Plant-based fibers
 - Crops nec
 - Cattle
 - Pigs
 - Poultry
 - Meat animals nec
 - Animal products nec
 - Raw milk
 - Wool, silk-worm cocoons
 - Manure (conventional treatment)
 - Manure (biogas treatment)
 - Products of forestry, logging and related services (02)
 - Fish and other fishing products; services incidental of fishing (05)
 - Anthracite
 - Coking Coal
 - Other Bituminous Coal
 - Sub-Bituminous Coal
 - Patent Fuel
 - Lignite/Brown Coal
 - BKB/Peat Briquettes
 - Peat
 - Crude petroleum and services related to crude oil extraction, excluding surveying
 - Natural gas and services related to natural gas extraction, excluding surveying
 - Natural Gas Liquids
 - Other Hydrocarbons
 - Uranium and thorium ores (12)
 - Iron ores
 - Copper ores and concentrates
 - Nickel ores and concentrates
 - Aluminium ores and concentrates
 - Precious metal ores 

## Exiobase final demand categories

In [27]:
final_demand_categories = Y.columns.get_level_values('category').unique()
print(f"\n{len(final_demand_categories)} final demand categories:")
for s in final_demand_categories:
    print(" -", s)


7 final demand categories:
 - Final consumption expenditure by households
 - Final consumption expenditure by non-profit organisations serving households (NPISH)
 - Final consumption expenditure by government
 - Net fixed capital formation
 - Changes in inventories
 - Changes in valuables
 - Exports: Total (fob)


## Endogeneisation du capital

### Avec endogénéisation du capital - méthode Andrieu et al.
(nowcasting from 2015 matrice)

In [33]:
DATA_FOLDER = "Data/"
YEAR = 2019
SYSTEM = "pxp"
FRANCE = "FR"

OUTPUT_FOLDER = Path("Results")

RE_IMPORT_EXIOBASE = False     # True = rebuild the table feathers from the ZIPs (also forces Kbar recomputation)
RECALCULATE_KBAR = False       # True = recompute Kbar even if its feather exists
ENDOGENEISER_CAPITAL = True
METHODE_KBAR = "nowcasting"    # or "from_x"

EXIO_PATH_ZIP = DATA_FOLDER + "Source files/IOT_2019_pxp.zip"

# nowcasting inputs
BASE_YEAR = 2015
KBAR_BASE_PATH = DATA_FOLDER + f"Source files/Kbar_exio_v3_6_{BASE_YEAR}{SYSTEM}.mat"

exio = importlib.reload(importlib.import_module("exiobase_io"))
d = exio.generate_exiobase(
    DATA_FOLDER, YEAR, SYSTEM,
    exio_path_zip=EXIO_PATH_ZIP,
    re_import_exiobase=RE_IMPORT_EXIOBASE,
    endogenise_capital=ENDOGENEISER_CAPITAL,
    kbar_method=METHODE_KBAR,
    recalculate_kbar=RECALCULATE_KBAR,
    kbar_base_path=KBAR_BASE_PATH,
    base_year=BASE_YEAR,
    # indicator=INDICATOR,
    # satellite="energy"
)

Z, A, Y, x, _, _, Kbar = (d[k] for k in ["Z", "A", "Y", "x", "F", "F_Y", "Kbar"])

✓ EXIOBASE files (newest) exist
✓ Using cached Kbar_nowcasting_2019_pxp_base2015.feather: (9800, 9800)


## Satellites

## Available satellites

In [41]:
satellite_list = ["air_emissions", "employment", "energy", "factor_inputs", "land", "material", "nutrients", "water"]
print(f"\n{len(satellite_list)} satellite tables available:")
for s in satellite_list:
    print(" -", s)


8 satellite tables available:
 - air_emissions
 - employment
 - energy
 - factor_inputs
 - land
 - material
 - nutrients
 - water


## Available indicators for each satellite

In [39]:
import zipfile

zip_path = f"{DATA_FOLDER}Source files/IOT_2019_ixi.zip"

with zipfile.ZipFile(zip_path, "r") as zf:
    for satellite in satellite_list:
        unit_path = f"{satellite}/unit.txt"
        print("=" * 60)
        print(f"Indicators available for: {satellite}")
        print("-" * 60)
        with zf.open(unit_path) as f:
            content = f.read().decode("utf-8", errors="replace")
        print(content)
        print()

Indicators available for: air_emissions
------------------------------------------------------------
stressor	unit
As - combustion - air	kg
B(a)P - combustion - air	kg
B(b)F - combustion - air	kg
B(k)F - combustion - air	kg
CH4 - combustion - air	kg
CO - combustion - air	kg
CO2 - combustion - air	kg
CO2_bio - combustion - air	kg
Cd - combustion - air	kg
Cr - combustion - air	kg
Cu - combustion - air	kg
HCB - combustion - air	kg
Hg - combustion - air	kg
Indeno - combustion - air	kg
N2O - combustion - air	kg
NH3 - combustion - air	kg
NMVOC - combustion - air	kg
NOx - combustion - air	kg
Ni - combustion - air	kg
PCB - combustion - air	kg
PCDD/F - combustion - air	kg
PM10 - combustion - air	kg
PM2_5 - combustion - air	kg
Pb - combustion - air	kg
SO2 - combustion - air	kg
SOx - combustion - air	kg
Se - combustion - air	kg
TSP - combustion - air	kg
Zn - combustion - air	kg
As - non combustion - Agglomeration plant - pellets - air	kg
As - non combustion - Agglomeration plant - sinter - air	kg

## Final energy use satellite extraction

In [43]:
INDICATOR = "Energy use - Final"
satellite = "energy"
F,F_Y = exio.load_sattelite_selection(DATA_FOLDER, YEAR, SYSTEM, INDICATOR, satellite)

- F is the energy use for each intermediate industry
- F_Y is the direct energy use of final demand categories

# Example : plotting final energy footprint Sankey

In [ ]:
from footprint_sankey import load_maps, compute_footprint_flows, make_sankey, save_sankey


satellite_mapping={r"^Energy use - Final": "Final energy use"}
maps = load_maps("Data/Concordance_Tables/concordance.xlsx")

title="France footprint Sankey"
unit = "TJ"

res = compute_footprint_flows(A, Y, x, F, F_Y, satellite_mapping,
    target_regions=["FR"], region_map=maps["region"], sector_map=maps["sector"],
    fd_map=maps["fd"], cons_map=maps["cons"], target_label="France",
    drop_empty_exports=True)
res["selected_stressors"], res["totals"], res["by_need"]


my_colors = {
    "Agriculture-food":       "#8AB17D",   # sage green
    "Energy industry":        "#F4A261",   # warm orange
    "Heavy industry":         "#457B9D",   # steel blue
    "Manufacturing industry": "#B392AC",   # dusty mauve
    "Services":               "#2A9D8F",   # teal
    "Transport services":     "#E76F51",   # coral
    "Other industries":       "#CED4DA",   # pale grey
    "negative":               "#9D4EDD",   # purple, clearly "different"
    "direct":                 "#495057",   # charcoal
    "final":                  "#264653",   # dark petrol blue
}

fig = make_sankey(
    res, title=title, unit=unit, min_industry_share=0.01,
    colors=my_colors,
    final_color=(60, 60, 60),        # color of Final energy use → Domestic/Imports
    node_color="rgba(70,70,70,0.8)", # color of all nodes
    height=1000, width=2500,
)
fig.show()

save_sankey(fig, "outputs/france_sankey.html")
save_sankey(fig, "outputs/france_sankey.png", scale=2)
save_sankey(fig, "outputs/france_sankey.pdf")


WindowsPath('outputs/france_sankey.pdf')

# [Exercice] Calculer les imports Chine --> EU27 pour chacun des 200 produits industriels d'Exiobase : quantifier en terme monétaire, d'énergie finale et de GES (CO2, N2O et CH4).

# [Projet] Quantifier un vecteur Y de demande finale pour les indicateurs utilisés dans les Decent Living Energy

## Indicateurs des DLS

### Main assumptions

#### Population

In what follows we will derive figures per capita, but sometimes we have to take assumption of age distribution in the country in question : Education, ICT devices

In [1]:
# Population from INSEE in central scenario of 2050 https://www.insee.fr/fr/statistiques/2418126
pop_tot = 71630 + 2400
population = {} # Population by age group in thousands
population["0_4"] = 3784 + 178 # thousands - hexogonal france + DROM
population["5_9"] = 3950 + 179 # thousands - hexogonal france + DROM
population["10_14"] = 4035 + 177 # thousands - hexogonal france + DROM
population["15_19"] = 4034 + 169 # thousands - hexogonal france + DROM
population["20_24"] = 3799 + 137
population["25_29"] = 3741 + 122
population["30_34"] = 3927 + 123
population["35_39"] = 4209 + 131
population["40_44"] = 4302 + 141
population["45_49"] = 4213 + 138

Share of women/men and population over 20 is used for nutrition needs (kcal/pers/day)

In [2]:
share_women = {} # Share of women from INSEE central projection in overall population (Hexognal France + oversea territory) https://www.insee.fr/fr/statistiques/2418126
share_women["0_4"] = (1831+87)/(population["0_4"])
share_women["5_9"] = (1925+88)/(population["5_9"])
share_women["10_14"] = (1965+87)/(population["10_14"])
share_women["15_19"] = (1966+83)/(population["15_19"])
share_women["20_24"] = (1879+68)/(population["20_24"])
share_women["25_29"] = (1883+64)/(population["25_29"])
share_women["30_34"] = (1984+67)/(population["30_34"])
share_women["35_39"] = (2123+72)/(population["35_39"])
share_women["40_44"] = (2167+76)/(population["40_44"])
share_women["45_49"] = (2123+74)/(population["45_49"])

total_women = 0.51*71630 + 0.535*2400 # Thousands

share_women_50_over =  total_women
pop_50_over = pop_tot
for age in  share_women :
    share_women_50_over -= share_women[age]*population[age]
    pop_50_over -= population[age]
share_women_50_over = share_women_50_over/pop_50_over

#### Households

In [3]:
capita_per_household_DLS = 4 # From DLE studies : https://www.sciencedirect.com/science/article/pii/S0959378020307512
capita_per_household_2020 = (10.8*1 + 9.5*2 + 3.9*3 + 3.5*4 + 1.7*5)/(10.8 + 9.5 + 3.9 + 3.5 + 1.7) # 5 and more are accounted for as 5
capita_per_household = (14.2*1 + 10.5*2 + 3.7*3 + 3.1*4 + 1.5*5)/(14.2 + 10.5 + 3.7 + 3.1 + 1.5) # 2050 in France in central scenario of household projection study of SDES and INSEE https://www.statistiques.developpement-durable.gouv.fr/projections-du-nombre-de-menages-horizon-2030-et-2050-une-croissance-portee-par-laugmentation-du
print(f"Here, we use capita per household projected by SDES and Insee in France in 2050 : {capita_per_household} capita/household. This figure was around {capita_per_household_2020} in 2020 and {capita_per_household_DLS} in DLS study")

Here, we use capita per household projected by SDES and Insee in France in 2050 : 2.0060606060606063 capita/household. This figure was around 2.17687074829932 in 2020 and 4 in DLS study


####  Some energy efficiency assumptions

In [15]:
COKE_NCV_PJ_PER_MT = 28  # 1 Mt coke → PJ for France in 2015 https://unstats.un.org/unsd/energy/yearbook/2015/08i.pdf

finaldemand_to_finalenergy_BEE = { # Best efficiency available in 2020 (supposed to be the overall efficiency in 2050) --> used to go from final energy in DLE study to final useful demand in KiNESYS (including efficiency)
    ("RSH", "PJ") : 1/3.22,  # efficiency of RSHEHPV3 in 2040 (not vintaged) : used in France in 2050 as the best option
    ("RLI", "PJ") : 1/2.76, # Current most efficient LED in stocks of 2019 : RLIFLE000
    ("RSC", "PJ") : 1/4.69, # efficiency of RSCEHPV2 in 2040 (not vintaged) : used in France in 2050 as the best option
}

#### Nutrition

##### Food

In [18]:
# From supplementary information of https://www.sciencedirect.com/science/article/pii/S0959378020307512#tblfn3, from Dietary Guidelines for Americans
daily_food_intake_age_sex = { # kcal/cap/day
    "0_4" : {
        "Males" : 1267,
        "Females" : 1200
    },
    "5_9" : {
        "Males" : 1600,
        "Females" : 1520
    },
    "10_14" : {
        "Males" : 2120,
        "Females" : 1920
    },
    "15_19" : {
        "Males" : 2760,
        "Females" : 2040
    },
    "20_49" : {
        "Males" : 2600,
        "Females" : 2033
    },
    "50_over" : {
        "Males" : 2300,
        "Females" : 1800
    }
}

average_food_intake = 0

for age in daily_food_intake_age_sex :
    if age in population :
        share_w = share_women[age]
        pop = population[age]
        # average_food_intake += (daily_food_intake_age_sex[age]["Males"]*population[age]*(1-share_women[age]) + daily_food_intake_age_sex[age]["Females"]*population[age]*share_women[age])
    elif age == "20_49" :
        share_w = 0
        pop = 0
        for age_share in ["20_24", "25_29", "30_34", "35_39", "40_44", "45_49"]:
            share_w += share_women[age_share]*population[age_share]
            pop += population[age_share]
        share_w = share_w/pop
    elif age == "50_over" :
        share_w = share_women_50_over
        pop = pop_50_over
    else :
        raise Warning(f"Age {age} is not treated properly")
    average_food_intake += (daily_food_intake_age_sex[age]["Males"]*pop*(1-share_w) + daily_food_intake_age_sex[age]["Females"]*pop*share_w)
average_food_intake = average_food_intake/pop_tot

print(f"Average food intake per person in 2050 taking into account Males/Women, age intakes needs : {average_food_intake} kcal/day/cap")
print(f"In supplementary data sheet given by Millward Hopkins, they give : 2089.86 kcal/day/cap, with 1944 for plant-based, 84 from non-meat animal-based, and 61 from meat")
kcal_per_person_per_day = average_food_intake

Average food intake per person in 2050 taking into account Males/Women, age intakes needs : 2080.863676887748 kcal/day/cap
In supplementary data sheet given by Millward Hopkins, they give : 2089.86 kcal/day/cap, with 1944 for plant-based, 84 from non-meat animal-based, and 61 from meat


##### Cooking

In [5]:
stove_per_household = 1
lifespan_stove = 15 # years (20 years in DLE study, this is from Material requirements study), 10 years in https://www.sciencedirect.com/science/article/abs/pii/S0959652619305955 from which we took energy intensity of manufacturing

In [ ]:
Final_energy_MJ_per_kg = 6 # Final energy MJ/kg (average figure for world) - FAO, Supplementary information of https://www.sciencedirect.com/science/article/pii/S0959378020307512?via%3Dihub, https://www.theconsciouschallenge.org/ecologicalfootprintbibleoverview/food-and-energy
kcal_per_g = 1.5 # Supplementary information of https://www.sciencedirect.com/science/article/pii/S0959378020307512?via%3Dihub
Final_energy_kJ_per_kcal = Final_energy_MJ_per_kg/kcal_per_g 
share_cooked_food = 0.5 # FAO, Supplementary information of https://www.sciencedirect.com/science/article/pii/S0959378020307512?via%3Dihub
max_energy_savings = 0.8  # But this assumes that we cook everything with a lid (couvercle). Also, this figure depends on the distribution Stove-top/oven/microwave. For stove only, 88%, oven 83%, microwave 0% https://pubs.acs.org/doi/suppl/10.1021/es102641n/suppl_file/es102641n_si_001.pdf  (supplementary information of https://pubs.acs.org/doi/10.1021/es102641n)
reasonable_energy_savings = 0.4 # They say 80% in Supplementary information of https://www.sciencedirect.com/science/article/pii/S0959378020307512?via%3Dihub, refering to an article by Cullen, Allwood and Borgstein (but in practice, difficult to put in place : cooking with a lid, non-metallic pots, etc.
# KiNESYS_energy_efficiency_France = 0.628730322
KiNESYS_energy_efficiency_World = 7937.40877/26963.05665 # Using this one because final energy consumption is given as a current value
# KiNESYS_best_efficiency = 0.7
kcal_to_GJ = 0.0000041868
DLE_cooking_GJ_perGJfood = Final_energy_kJ_per_kcal/(1000000*kcal_to_GJ)*share_cooked_food*(1-reasonable_energy_savings)*KiNESYS_energy_efficiency_World
RCK_PJ_per_capita = kcal_per_person_per_day*365*0.0000041868*DLE_cooking_GJ_perGJfood/1000000
print(f'Cooking energy per capita per year (PJ/cap/year) : {RCK_PJ_per_capita}') # PJ/cap/year : "useful" energy

Cooking energy per capita per year (PJ/cap/year) : 2.683041327945742e-07


##### Refrigerator and freezer

In [8]:
refrigerator_per_household = 1 # In https://www.sciencedirect.com/science/article/pii/S0959378020307512, for indirect energy, they consider 2 (one refrigerator and one freeze)
lifespan_refrigerator = 10 # In https://www.sciencedirect.com/science/article/pii/S0959378020307512, they consider 14

In [ ]:
Final_energy_kWh_per_year = 120
Final_energy_PJ_per_capita_per_year = 120*3.6*1e-9*refrigerator_per_household/capita_per_household
consumption_reduction_compared_to_today = 0.8 # Used to increase the need - Only technology-based considering best available fridges, in Supplementary information of https://www.sciencedirect.com/science/article/pii/S0959378020307512?via%3Dihub, refering to an article by Cullen, Allwood and Borgstein
KiNESYS_energy_efficiency_World_2019 = 2741.89335294386/3502.13920666253 # Var_FOut/Var_FIn

RRF_PJ_per_capita_per_year = Final_energy_PJ_per_capita_per_year * KiNESYS_energy_efficiency_World_2019 / (1-consumption_reduction_compared_to_today) # Useful energy consumption as defined in KiNESYS, without accounting for technological change
RRF_PJ_per_capita_per_year
print(f'Home refrigeration energy per capita per year (PJ/cap/year) : {RRF_PJ_per_capita_per_year}') # PJ/cap/year : "useful" energy

Home refrigeration energy per capita per year (PJ/cap/year) : 8.42998603992238e-07


#### Shelter & living conditions

In [10]:
m2_per_capita = 10 + 20/capita_per_household # m2
print(f"m2_per_capita: {m2_per_capita} m2/capita")

m2_per_capita: 19.96978851963746 m2/capita


##### Direct energy

In [ ]:
share_cooling_2050_NZE = 13.04/(13.04+389.83) # from KiNESYS, have to take an hypothesis because not given in suplementary materials of https://www.sciencedirect.com/science/article/pii/S0959378020307512 
FRA_thermalconfort_per_capita_per_year_millward = 0.682833932 # GJ/capita/year, with best performing house (from Millward-Hopkins supplementary excel data sheet)
min_thermalconfort_millward = 0.291518935 # This will be used to induce heating and cooling needs in commercial buildings (education, healthcare and other commercial). Maximal value among all countries
max_thermalconfort_millward = 0.832447991 # This will be used to induce heating and cooling needs in commercial buildings (education, healthcare and other commercial). Minimal value among all countries



final_energy_per_capita_per_year = FRA_thermalconfort_per_capita_per_year_millward
final_cooling_per_capita_per_year = final_energy_per_capita_per_year*share_cooling_2050_NZE
final_heating_per_capita_per_year = final_energy_per_capita_per_year*(1-share_cooling_2050_NZE)
RSC_per_capita_per_year = final_cooling_per_capita_per_year / finaldemand_to_finalenergy_BEE[("RSC", "PJ")] / 1e+6
RSH_per_capita_per_year = final_heating_per_capita_per_year / finaldemand_to_finalenergy_BEE[("RSH", "PJ")] / 1e+6


print(f'Heating energy per capita per year (PJ/capita/year) : {RSH_per_capita_per_year}') # PJ/cap/year : "useful" energy
print(f'Cooling energy per capita per year (PJ/capita/year) : {RSC_per_capita_per_year}') # PJ/cap/year : "useful" energy

Heating energy per capita per year (PJ/capita/year) : 2.1275574465987127e-06
Cooling energy per capita per year (PJ/capita/year) : 1.0365746886013652e-07


##### Illumination (Residential)

In [16]:
A_illuminated_area = m2_per_capita*0.33
t_illumination_time = 6*3600*365 # seconds/year (6 hours/day)
E_lighting_need = 125 # lm/m2
efficacy_LED_current = 800/5.8 # in DLS, 150 (here 138) lm/W, from https://www1.eere.energy.gov/buildings/publications/pdfs/ssl/2012_LED_Lifecycle_Report.pdf
energy_lighting_per_capita_per_year = t_illumination_time*A_illuminated_area*E_lighting_need/efficacy_LED_current/1e+6
energy_lighting_per_capita_per_year # MJ/cap/year
RLI_per_capita_per_year = energy_lighting_per_capita_per_year / 1e+9 / finaldemand_to_finalenergy_BEE[("RLI", "PJ")]
print(f'Lighting energy per capita per year (PJ/capita/year) : {RLI_per_capita_per_year}') # PJ/cap/year

Lighting energy per capita per year (PJ/capita/year) : 1.2995444021601204e-07


##### House construction

#### Hygiene

##### Water supply

In [19]:
water_need_drinking = 1.5 # mL/kcal of food
water_need_sanitation = 20 # L/day
water_need_bathing = 15 # L/day
water_need_foodpreparation = 10 # L/day

total_water_need_per_day_per_capita = water_need_drinking*kcal_per_person_per_day/1000+water_need_sanitation+water_need_bathing+water_need_foodpreparation
print(f"Water need (L/day/capita): {total_water_need_per_day_per_capita}, in supplementary data sheet from Millward-Hopkins, 47.6123 L/day/person in France")
print(f"UN recommandation (L/day/capita): 50-100 L/day/cap")
total_water_need_per_year_per_capita = total_water_need_per_day_per_capita*365 # L/year

Water need (L/day/capita): 48.121295515331624, in supplementary data sheet from Millward-Hopkins, 47.6123 L/day/person in France
UN recommandation (L/day/capita): 50-100 L/day/cap


##### Water heating

In [21]:
hot_water_need_bathing = 15 # L/cap/day
hot_water_need_sanitation = 0.25*20 # L/cap/day

hot_water_need_per_capita_per_year = (hot_water_need_bathing + hot_water_need_sanitation)*365
print(f"Hot water need (L/year/capita): {hot_water_need_per_capita_per_year}")
C_p = 4184 # J/(kg.K)
rho = 0.997 # kg/L
T_out = 50
T_in = 12.5 # average temperature in France

RHW_PJ_per_capita = C_p*hot_water_need_per_capita_per_year*rho*(T_out - T_in)/1e+15 # Calculation used in https://www.sciencedirect.com/science/article/pii/S0959378020307512, but we don't take into account the efficiency to get useful energy
print(f"Hot water need (PJ/year/capita): {RHW_PJ_per_capita}")

Hot water need (L/year/capita): 7300.0
Hot water need (PJ/year/capita): 1.14193389e-06


##### Waste management

May be already included when using exiobase approach ?

In [ ]:
dict_waste_quantity = { # From https://pubs.acs.org/doi/10.1021/acs.est.3c03957#_i23
    "waste_open_dump" : 89, # kg/cap/year
    "waste_open_burning" : 30, # kg/cap/year
    "waste_unsanitary_landfill" : 68, # kg/cap/year
    "waste_biowaste" : 14, # kg/cap/year
    "waste_sanitary_landfill" : 70, # kg/cap/year
    # residential_wastewater = 18 # m3/cap/year, already accounted for in "Water supply" in principle
}

#### Clothing

In [ ]:
data = {
    "Item": ["Tops", "Bottoms", "Jackets", "Jumpers", "Underwear", "Shoes"],
    "Weight_kg": [0.25, 0.5, 0.5, 0.5, 0.05, 0.6],
    "Lifetime_years": [112.5/365, 300/365, 562.5/365, 337.5/365, 125/365, 365/365],  # Converted to years : days of wearing
    "Days_per_wash": [3, 15, 25, 15, 1.5, 60],
    "Embodied_energy_MJ": [62, 172, 111, 172, 12, 133],
    "Potential_energy_saving_pct": [0.76, 0.76, 0.44, 0.44, 0.76, 0.58], # Only used for Agriculture which is in Final energy
    "Material": ["cotton", "cotton", "wool", "wool", "cotton", "rubber"] # Rubber considered natural here, but in https://pubs.acs.org/doi/10.1021/acs.est.3c03957 they consider synthetic
}

df_clothes = pd.DataFrame(data)

df_clothes["kg_per_year_wash"] = 365/df_clothes["Days_per_wash"]*df_clothes["Weight_kg"]
total_weight_washing_per_year = df_clothes["kg_per_year_wash"].sum()
total_weight_newclothes_per_year = ((1 / df_clothes['Lifetime_years']) * df_clothes["Weight_kg"]).sum()
df_clothes['Final_energy_per_year_no_saving'] = (1 / df_clothes['Lifetime_years']) * df_clothes['Embodied_energy_MJ']
df_clothes['Final_energy_per_year_saving'] = (1 / df_clothes['Lifetime_years']) * df_clothes['Embodied_energy_MJ'] * (1-df_clothes['Potential_energy_saving_pct'])

print(f"Total weight new clothes per year (without 20% increase of towels) : {total_weight_newclothes_per_year} (in Supplementary data sheet given by Millward Hopkins, total is 3.63676 kg/year/capita)")
print(f"Total weight washed clothes per year (without 20% increase of towels) : {total_weight_washing_per_year}")

Additional_factor_othertextiles = 0.2 # 20% to account for towels and beddings https://www.sciencedirect.com/science/article/pii/S0959378020307512

Total weight new clothes per year (without 20% increase of towels) : 3.03062962962963 (in Supplementary data sheet given by Millward Hopkins, total is 3.63676 kg/year/capita)
Total weight washed clothes per year (without 20% increase of towels) : 77.86666666666667


##### Clothes

##### Clothes washing

#### Healthcare

In [ ]:
floor_space_per_hospital_bed = 200 # m2/bed
number_hospital_bed_per_capita = 8/1000 # 8 beds for 1000 citizens
m2_healthcare_per_capita = floor_space_per_hospital_bed*number_hospital_bed_per_capita
print(f"m2 of healthcare per capita : {m2_healthcare_per_capita}")
# upscaling_factor_other_energy_building = 2

m2 of healthcare per capita : 1.6
Scaling factor from Other uses (equipment, ventilation) to Lighting: 1.4456521739130435


#### Education

In [29]:
floor_space_per_pupil = 10 # m2/pupil

share_pupil_population = (population["5_9"] + population["10_14"] + population["15_19"] )/pop_tot
print(f"Share of pupils (5-19 years old) in French population in central scenario of INSEE - 2017 : {share_pupil_population}")

m2_education_per_capita = floor_space_per_pupil*share_pupil_population # m2/capita
print(f"m2 of education per capita : {m2_education_per_capita}")

Share of pupils (5-19 years old) in French population in central scenario of INSEE - 2017 : 0.16944481966770228
m2 of education per capita : 1.6944481966770228


#### Communication and information

##### Phone

In [ ]:
number_phone_per_adult = 1

share_adult_population = (pop_tot - population["0_4"] - population["5_9"])/pop_tot # Considered over 10
print(f"Share of \"adults\" (over 10 years old) in French population in central scenario of INSEE - 2017 : {share_adult_population}")

number_phone_per_capita = number_phone_per_adult * share_adult_population
print(f"Number of phone per adult : {number_phone_per_adult}")
print(f"Number of phone per capita : {number_phone_per_capita}")
lifetime_phone = 2.5 # We use the same as DLS materials : https://pubs.acs.org/esthag/article/57/38/14206/399422/Material-Requirements-of-Decent-Living-Standards

Share of "adults" (over 10 years old) in French population in central scenario of INSEE - 2017 : 0.8907064703498582
We use new reference flow : 0.3562825881399433 unit/(cap.year) instead of previous value in DLS materials study : 0.328


##### Computer

In [31]:
number_computer_per_household = 1

number_computer_per_capita = number_computer_per_household/capita_per_household # https://www.sciencedirect.com/science/article/pii/S0959652611000801
print(f"Number of computer per capita : {number_computer_per_capita}")
lifetime_computer = 4.25 # We use the same as DLS materials : https://pubs.acs.org/esthag/article/57/38/14206/399422/Material-Requirements-of-Decent-Living-Standards

Number of computer per capita : 0.49848942598187307


##### Networks and infrastructure

In [ ]:
number_router_household = 1 

number_router_per_capita = number_router_household / capita_per_household
print(f"Number of router per capita : " {number_router_per_capita})
lifetime_router = 6 # We use the same as DLS materials

# To get ratio to get tier2 (Fixed and mobile networks) and tier3 energy requirements (Datacenters)
MIPS_LCA_ratio_TIER2_box = 1/0.59/0.828 # Tableau 75 and 72 de https://www.arcep.fr/uploads/tx_gspublication/etude-numerique-environnement-ademe-arcep-volet02_janv2022.pdf
MIPS_LCA_ratio_TIER2and3_TIER2 = (0.115+0.036+0.005+0.063)/(0.005+0.063) # Tableau 68 de https://www.arcep.fr/uploads/tx_gspublication/etude-numerique-environnement-ademe-arcep-volet02_janv2022.pdf

#### Mobility

##### Passenger kilometers per mode

###### Overall mobility

In [ ]:
LD_base = 189 # Long distance people/km2
f_variable = 0.5
f_fixed = 1 - f_variable
MOB_base_rural = 10000 # km/year/capita
MOB_base_urban = 7000

### Data from World Bank
Agricultural_land_area = 282972 # m2, France, 2023 https://data.worldbank.org/indicator/AG.LND.AGRI.K2?name_desc=false
Urban_land_area = 23031 # m2, France, 2015 https://data.worldbank.org/indicator/AG.LND.TOTL.UR.K2?name_desc=false
Total_land_area = 538950 # m2, France, 2023 https://data.worldbank.org/indicator/AG.LND.TOTL.K2?name_desc=false
Popuation_density = 127 # people/m2, France, 2023 https://data.worldbank.org/indicator/EN.POP.DNST?name_desc=false

share_rural_pop = 0.21 # %, France, 2024 https://data.worldbank.org/indicator/SP.RUR.TOTL.ZS?name_desc=false
share_urban_pop = 1 - share_rural_pop

In [ ]:
LD = Popuation_density * Total_land_area/ (Urban_land_area + Agricultural_land_area) # Long distance
print(f'LD is {LD} people /km2')
MOB_rural = MOB_base_rural * (f_fixed + f_variable * LD_base/LD)
print(f'Rural mobility is {MOB_rural} km/capita/year')
MOB_urban = MOB_base_urban * (f_fixed + f_variable * LD_base/LD)
print(f'Urban mobility is {MOB_urban} km/capita/year')
MOB = MOB_rural * share_rural_pop + MOB_urban * share_urban_pop
print(f'Average mobility is {MOB} km/capita/year, in supplementary data sheet given by Millward-Hopkins, 6613 km/pers/year')

LD is 223.67966980715875 people /km2
Rural mobility is 9224.79164429523 km/capita/year
Urban mobility is 6457.354151006661 km/capita/year
Average mobility is 7038.51602459726 km/capita/year, in supplementary data sheet given by Millward-Hopkins, 6613 km/pers/year


###### Modal share - ADEME S2

In [35]:
# From CLEVER, 2020
ratio_2w3w = 16.585/(16.585 + 787.341966928483) # ratio Gpkm 2w+3w over 2w+3w+car. 2020 CLEVER

## Distance by person by mode in 2050 in S2
walk_ADEME_S2 = 1.06 # km/cap/day
bike_ADEME_S2 = 3.4 # km/cap/day
rail_ADEME_S2 = 7.9 # km/cap/day
bus_ADEME_S2 = 2.6 # km/cap/day
car_ADEME_S2 = 19.2 # km/cap/day. Also 2w
plane_ADEME_S2 = 6.1 # km/cap/day # Check if this includes all international travel ? Or half of residents to be territorial-based ?

total_ADEME_S2 = walk_ADEME_S2 + bike_ADEME_S2 + rail_ADEME_S2 + bus_ADEME_S2 + car_ADEME_S2 + plane_ADEME_S2

air_mobility = MOB * plane_ADEME_S2 / total_ADEME_S2 # km/capita/year
TAI_PASS_LOCAL = 125.867029 # It will be multiplied by 2 afterwards to distribute overall air travel km between international and domestic
TAD_PASS_LOCAL = 48.712294

factor_for_air_mobility = 2.0

walk_mobility =  MOB * walk_ADEME_S2 / total_ADEME_S2 # km/capita/year
bike_mobility =  MOB * bike_ADEME_S2 / total_ADEME_S2 # km/capita/year
nonmotorised_mobility = walk_mobility + bike_mobility

air_mobility_international = air_mobility*TAI_PASS_LOCAL*factor_for_air_mobility/(TAI_PASS_LOCAL*factor_for_air_mobility+TAD_PASS_LOCAL) # km/capita/year. Using sufficiency scenario of ADEME. We used to divide by 2 to account only for territorial consumption, but we now account for them.
air_mobility_domestic = air_mobility*TAD_PASS_LOCAL/(TAI_PASS_LOCAL*factor_for_air_mobility+TAD_PASS_LOCAL) # km/capita/year. Using sufficiency scenario of ADEME.
rail_mobility = MOB * rail_ADEME_S2 / total_ADEME_S2  # km/capita/year
bus_mobility =  MOB * bus_ADEME_S2 / total_ADEME_S2  # km/capita/year
car_mobility =  MOB * car_ADEME_S2 / total_ADEME_S2 * (1 - ratio_2w3w) # km/capita/year
w2_mobility =  MOB * car_ADEME_S2 / total_ADEME_S2 * (ratio_2w3w) # km/capita/year

print(f"Non-motorised mobility: {nonmotorised_mobility} km/capita/year, in supplementary data sheet given by Millward-Hopkins, 1251 km/pers/year")
print(f"Air mobility - international: {air_mobility_international} km/capita/year, in supplementary data sheet given by Millward-Hopkins, 0.33 A/R per year per person (for all aviation)")
print(f"Air mobility - domestic: {air_mobility_domestic} km/capita/year, in supplementary data sheet given by Millward-Hopkins, 0.33 A/R per year per person (for all aviation)")
print(f"Rail mobility: {rail_mobility} km/capita/year")
print(f"Bus mobility: {bus_mobility} km/capita/year")
print(f"Car mobility: {car_mobility} km/capita/year, in supplementary data sheet given by Millward-Hopkins, 1072 km/pers/year (without considering no 2w)")
print(f"2w mobility: {w2_mobility} km/capita/year")

Non-motorised mobility: 779.7263156906056 km/capita/year, in supplementary data sheet given by Millward-Hopkins, 1251 km/pers/year
Air mobility - international: 893.5363190896201 km/capita/year, in supplementary data sheet given by Millward-Hopkins, 0.33 A/R per year per person (for all aviation)
Air mobility - domestic: 172.9055028190559 km/capita/year, in supplementary data sheet given by Millward-Hopkins, 0.33 A/R per year per person (for all aviation)
Rail mobility: 1381.129572635826 km/capita/year
Bus mobility: 454.54897327255037 km/capita/year
Car mobility: 3287.4213082307624 km/capita/year, in supplementary data sheet given by Millward-Hopkins, 1072 km/pers/year (without considering no 2w)
2w mobility: 69.24803285883985 km/capita/year


In [36]:
mobility_dict_Gpkm = { # Gpkm/capita/year
    "TAI_PASS_LOCAL" : air_mobility_international/1e+9,
    "TAD_PASS_LOCAL" : air_mobility_domestic/1e+9,
    "TTP" : rail_mobility/1e+9,
    "Trd_Bus" : bus_mobility/1e+9,
    "Trd_Car" : car_mobility/1e+9,
    "Trd_2w" : w2_mobility/1e+9,
    "Bike" : bike_mobility/1e+9, # No direct energy in KiNESYS, but manufacturing taken into account
    "Walk" : walk_mobility/1e+9 # No direct energy in KiNESYS, but manufacturing taken into account
}

#### Other (missed commercial buildings, freight and energy system)--> but this may already be accounted for using Exiobase approach : we have to discuss this. It depends on the boundaries of the approach

##### Commercial building

##### Freight

##### Power supply